Copy the code below into a file named generate_insureco_data.py.

Run it in your terminal: python generate_insureco_data.py.

It will create a folder insureco_data/ with 6 CSV files.

Upload these to Databricks/ADLS as per Day 1 instructions.

In [0]:
import csv
import random
import os
from datetime import datetime, timedelta

# ================= CONFIGURATION =================
SCALE_FACTOR = 1000  # 1000 = ~50k customers. Change to 100 for small test, 5000 for huge.
OUTPUT_DIR = "insureco_data"
SEED = 42

# ================= HELPERS =================
random.seed(SEED)

def random_date(start_year, end_year):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    delta = end - start
    random_days = random.randint(0, delta.days)
    return (start + timedelta(days=random_days)).strftime("%Y-%m-%d")

def random_amount(min_val, max_val):
    return round(random.uniform(min_val, max_val), 2)

# ================= DATA GENERATORS =================

def generate_agents(count):
    data = []
    branches = ["Mumbai-Central", "Delhi-CP", "Bangalore-MG", "Chennai-T Nagar", "Hyderabad-Banjara", "Pune-FC", "Kolkata-Park"]
    regions = ["West", "North", "South", "East"]
    for i in range(1, count + 1):
        agent_id = f"AGT{i:05d}"
        name = f"Agent {random.choice(['Smith', 'Kumar', 'Reddy', 'Singh', 'Gupta'])} {i}"
        branch = random.choice(branches)
        region = random.choice(regions)
        hire_date = random_date(2010, 2023)
        data.append([agent_id, name, branch, region, hire_date])
    return data

def generate_customers(count):
    data = []
    cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad", "Pune", "Kolkata", "Jaipur"]
    states = ["MH", "DL", "KA", "TN", "TS", "MH", "WB", "RJ"]
    for i in range(1, count + 1):
        cust_id = f"CUST{i:06d}"
        name = f"Customer {i}"
        dob = random_date(1950, 2000)
        gender = random.choice(["M", "F", "O"])
        city = random.choice(cities)
        state = random.choice(states)
        signup = random_date(2015, 2024)
        data.append([cust_id, name, dob, gender, city, state, signup])
    return data

def generate_policies(count, customer_ids, agent_ids):
    data = []
    types = ["Auto", "Health", "Life", "Home"]
    statuses = ["Active", "Expired", "Cancelled", "Lapsed"]
    for i in range(1, count + 1):
        pol_id = f"POL{i:07d}"
        cust_id = random.choice(customer_ids)
        agent_id = random.choice(agent_ids)
        p_type = random.choice(types)
        start = random_date(2018, 2023)
        end = random_date(2024, 2030)
        premium = random_amount(5000, 100000)
        status = random.choices(statuses, weights=[60, 20, 10, 10])[0] # 60% Active
        data.append([pol_id, cust_id, agent_id, p_type, start, end, premium, status])
    return data

def generate_payments(count, policy_ids):
    data = []
    modes = ["CreditCard", "NetBanking", "AutoDebit", "UPI"]
    for i in range(1, count + 1):
        pay_id = f"PAY{i:08d}"
        pol_id = random.choice(policy_ids)
        date = random_date(2020, 2024)
        amount = random_amount(500, 5000)
        mode = random.choice(modes)
        is_late = random.choices(["true", "false"], weights=[15, 85])[0]
        data.append([pay_id, pol_id, date, amount, mode, is_late])
    return data

def generate_claims(count, policy_ids):
    data = []
    types = ["Accident", "Surgery", "Theft", "Fire", "DeathBenefit", "NaturalDisaster"]
    statuses = ["Filed", "UnderReview", "Approved", "Rejected"]
    for i in range(1, count + 1):
        claim_id = f"CLM{i:07d}"
        pol_id = random.choice(policy_ids)
        date = random_date(2020, 2024)
        amount = random_amount(1000, 500000)
        ctype = random.choice(types)
        status = random.choices(statuses, weights=[20, 30, 40, 10])[0]
        fraud = random.choices(["true", "false"], weights=[5, 95])[0]
        data.append([claim_id, pol_id, date, amount, ctype, status, fraud])
    return data

def generate_risk(count, policy_ids):
    data = []
    categories = ["Low", "Medium", "High"]
    for i in range(1, count + 1):
        pol_id = random.choice(policy_ids)
        score = random.randint(1, 100)
        if score < 40: cat = "Low"
        elif score < 70: cat = "Medium"
        else: cat = "High"
        date = random_date(2020, 2024)
        data.append([pol_id, score, cat, date])
    return data

def write_csv(filename, headers, data):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(data)
    print(f"✅ Generated {filepath} with {len(data)} rows")

# ================= EXECUTION =================
if __name__ == "__main__":
    print(f"🚀 Starting InsureCo Data Generator (Scale: {SCALE_FACTOR})...")
    
    # 1. Base Tables
    agents = generate_agents(1 * SCALE_FACTOR)
    customers = generate_customers(50 * SCALE_FACTOR)
    
    # Extract IDs for relationships
    agent_ids = [a[0] for a in agents]
    customer_ids = [c[0] for c in customers]
    
    # 2. Child Tables
    policies = generate_policies(200 * SCALE_FACTOR, customer_ids, agent_ids)
    policy_ids = [p[0] for p in policies]
    
    payments = generate_payments(500 * SCALE_FACTOR, policy_ids)
    claims = generate_claims(100 * SCALE_FACTOR, policy_ids)
    risk = generate_risk(200 * SCALE_FACTOR, policy_ids)
    
    # 3. Write Files
    write_csv("agents.csv", ["agent_id", "name", "branch", "region", "hire_date"], agents)
    write_csv("customers.csv", ["customer_id", "name", "dob", "gender", "city", "state", "signup_date"], customers)
    write_csv("policies.csv", ["policy_id", "customer_id", "agent_id", "policy_type", "start_date", "end_date", "premium_amount", "status"], policies)
    write_csv("premium_payments.csv", ["payment_id", "policy_id", "payment_date", "amount", "payment_mode", "is_late"], payments)
    write_csv("claims.csv", ["claim_id", "policy_id", "claim_date", "claim_amount", "claim_type", "claim_status", "fraud_flag"], claims)
    write_csv("underwriting_risk.csv", ["policy_id", "risk_score", "risk_category", "evaluated_date"], risk)
    
    print("🎉 All done! Check the 'insureco_data' folder.")

Run Inside Databricks (Skip Upload)

In [0]:
# Databricks Notebook Cell 1: Setup Volume Path
# Ensure you ran the Day 1 SQL to create catalog/schema/volume
volume_path = "/Volumes/insurance_dev/raw_files/landing"

# Databricks Notebook Cell 2: Run the Generator Logic (Simplified for Spark)
# We'll use Spark to generate data directly for massive scale without Python loops

from pyspark.sql.functions import expr, col, lit, round as spark_round, rand, when
from pyspark.sql.types import StringType, DoubleType, IntegerType

# Helper to generate IDs
def gen_id(prefix, count):
    return [f"{prefix}{i:07d}" for i in range(1, count+1)]

# 1. Agents (1k)
agents_df = spark.range(1, 1001).withColumn("agent_id", expr("concat('AGT', lpad(id, 5, '0'))")) \
    .withColumn("name", expr("concat('Agent ', id)")) \
    .withColumn("branch", expr("element_at(array('Mumbai', 'Delhi', 'Bangalore'), cast(rand()*3 as int)+1)")) \
    .withColumn("region", expr("element_at(array('West', 'North', 'South'), cast(rand()*3 as int)+1)")) \
    .withColumn("hire_date", expr("date_sub(current_date(), cast(rand()*3000 as int))")) \
    .select("agent_id", "name", "branch", "region", "hire_date")

agents_df.write.mode("overwrite").csv(f"{volume_path}/agents", header=True)

# 2. Customers (50k)
customers_df = spark.range(1, 50001).withColumn("customer_id", expr("concat('CUST', lpad(id, 6, '0'))")) \
    .withColumn("name", expr("concat('Customer ', id)")) \
    .withColumn("dob", expr("date_sub(current_date(), cast(rand()*20000 as int))")) \
    .withColumn("gender", expr("element_at(array('M', 'F', 'O'), cast(rand()*3 as int)+1)")) \
    .withColumn("city", expr("element_at(array('Mumbai', 'Delhi', 'Bangalore', 'Chennai'), cast(rand()*4 as int)+1)")) \
    .withColumn("state", expr("element_at(array('MH', 'DL', 'KA', 'TN'), cast(rand()*4 as int)+1)")) \
    .withColumn("signup_date", expr("date_sub(current_date(), cast(rand()*2000 as int))")) \
    .select("customer_id", "name", "dob", "gender", "city", "state", "signup_date")

customers_df.write.mode("overwrite").csv(f"{volume_path}/customers", header=True)

# 3. Policies (200k) - Links Customers & Agents
# We need to collect IDs to join them randomly
agent_ids = [r[0] for r in agents_df.select("agent_id").collect()]
cust_ids = [r[0] for r in customers_df.select("customer_id").collect()]

policies_df = spark.range(1, 200001).withColumn("policy_id", expr("concat('POL', lpad(id, 7, '0'))")) \
    .withColumn("customer_id", expr("element_at(array('" + "','".join(cust_ids[:100]) + "'), cast(rand()*100 as int)+1)")) \
    .withColumn("agent_id", expr("element_at(array('" + "','".join(agent_ids[:100]) + "'), cast(rand()*100 as int)+1)")) \
    .withColumn("policy_type", expr("element_at(array('Auto', 'Health', 'Life', 'Home'), cast(rand()*4 as int)+1)")) \
    .withColumn("start_date", expr("date_sub(current_date(), cast(rand()*1000 as int))")) \
    .withColumn("end_date", expr("date_add(current_date(), cast(rand()*1000 as int))")) \
    .withColumn("premium_amount", expr("cast(rand()*100000 as decimal(10,2))")) \
    .withColumn("status", expr("element_at(array('Active', 'Expired', 'Cancelled'), cast(rand()*3 as int)+1)")) \
    .select("policy_id", "customer_id", "agent_id", "policy_type", "start_date", "end_date", "premium_amount", "status")

policies_df.write.mode("overwrite").csv(f"{volume_path}/policies", header=True)

print("✅ Data generated directly in Volume!")